In [ ]:
# Online Retail II Customer Sales Analysis

## Data Quality Assessment

Author: Yoonju Song

Date: July 2026

---

### Objectives

The purpose of this notebook is to evaluate the quality of the Online Retail II dataset before performing exploratory data analysis and business analysis.

The assessment includes:

- Dataset overview
- Data types
- Missing values
- Duplicate records
- Invalid values
- Initial observations


In [7]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("ggplot")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [8]:
df = pd.read_csv("../data/raw/online_retail_II.csv")

In [9]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [10]:
df.shape

(1067371, 8)

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 65.1 MB


## 1. Dataset Overview

The dataset contains **1,067,370 transaction records** and **8 columns**.

Initial observations:

- Invoice, StockCode, Quantity, InvoiceDate and Price contain no missing values.
- Description contains missing values.
- Data cleaning will be required before performing business analysis

## 2. Missing Values

In [13]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [14]:
missing = df.isnull().sum()
missing

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [15]:
missing_percentage = (
    df.isnull().sum()/len(df)*100
)

missing_percentage

Invoice         0.000000
StockCode       0.000000
Description     0.410541
Quantity        0.000000
InvoiceDate     0.000000
Price           0.000000
Customer ID    22.766873
Country         0.000000
dtype: float64

In [16]:
missing_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": (
        df.isnull().sum()/len(df)*100
    ).round(2)
})
missing_summary

,Missing Count,Missing Percentage
Invoice,0,0.00
StockCode,0,0.00
Description,4382,0.41
Quantity,0,0.00
InvoiceDate,0,0.00
Price,0,0.00
Customer ID,243007,22.77
Country,0,0.00


## Interpretation
Customer ID has a relatively high percentage of missing values. which may affect customer-level analysis such as RFM segmentation.
Description contains only a small proportion of missing values and is expected to have minimal impact on the overall analysis.

## 3. Duplicate Records

This section identifies duplicate records in the dataset to assess data consistency before further analysis.

In [17]:
duplicate_count = df.duplicated().sum()
duplicate_count

np.int64(34335)

In [18]:
duplicate_percentage = (
    duplicate_count / len(df) * 100
).round(2)

print(f"Duplicate_Records : {duplicate_count:,}")
print(f"Duplicate_Percentage : {duplicate_percentage}%")

Duplicate_Records : 34,335
Duplicate_Percentage : 3.22%


## Interpretation
Duplicate records were identified in the dataset.
These records will be further investigated to determine whether they represent true duplicate transactions or valid repeated purchases before deciding whether removal is appropriate. 

## 4. Data Type

This section reviews the data types of each column to emsire they are appropriate for analysis

In [19]:
df.dtypes

Invoice            str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
Price          float64
Customer ID    float64
Country            str
dtype: object

## Interpretation

Most column have appropriate data types for analysis.

However, the **InvoiceData** column is currently stored as a string (object) and should be converted to a datetime format.

The **Customer ID** column is stored as a float because  it contains missing values. This data type will be reviewed during the data cleaning stage.


In [20]:
df["InvoiceDate"].head()

0    2009-12-01 07:45:00
1    2009-12-01 07:45:00
2    2009-12-01 07:45:00
3    2009-12-01 07:45:00
4    2009-12-01 07:45:00
Name: InvoiceDate, dtype: str

## 5. Quantity Validation

This section examines whether the Quantity column contatins invalid or negative values.

In [21]:
negative_quantity = df[df["Quantity"] <0]
negative_quantity.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
178,C489449,22087,PAPER BUNTING WHITE LACE,-12,2009-12-01 10:33:00,2.95,16321.0,Australia
179,C489449,85206A,CREAM FELT EASTER EGG BASKET,-6,2009-12-01 10:33:00,1.65,16321.0,Australia
180,C489449,21895,POTTING SHED SOW 'N' GROW SET,-4,2009-12-01 10:33:00,4.25,16321.0,Australia
181,C489449,21896,POTTING SHED TWINE,-6,2009-12-01 10:33:00,2.10,16321.0,Australia
182,C489449,22083,PAPER CHAIN KIT RETRO SPOT,-12,2009-12-01 10:33:00,2.95,16321.0,Australia


In [22]:
print(f"Returned Transactions: {len(negative_quantity):,}")

Returned Transactions: 22,950


## Interpretation

A total of **22,950 transactions** have negative quantities.

Futher insepection shows that these transactions are associated with invoice numbers beginning with the letter **"C"**, indicating cancelled or returned orders.

These records are valid business events rather than data quality errors and should be handled carefully depending on the business objective during the data cleaning stage.

## 6. Price Validation

This section evaluates whether the **Price** column contains zero or negative values that may affect sales analysis.

In [24]:
df[df["Price"] <= 0].head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
263,489464,21733,85123a mixed,-96,2009-12-01 10:52:00,0.0,NaN,United Kingdom
283,489463,71477,short,-240,2009-12-01 10:52:00,0.0,NaN,United Kingdom
284,489467,85123A,21733 mixed,-192,2009-12-01 10:53:00,0.0,NaN,United Kingdom
470,489521,21646,NaN,-50,2009-12-01 11:44:00,0.0,NaN,United Kingdom
3114,489655,20683,NaN,-44,2009-12-01 17:26:00,0.0,NaN,United Kingdom


In [25]:
zero_price = df[df["Price"] <=0]

print(f"Zero or Negative Price Records: {len(zero_price):,}") 

Zero or Negative Price Records: 6,207


### Interpretation

Transactions with zero or negative price require further investigation.

These records may represent free promotional items, product samples, manual adjustments, or data entry issues.

There impact will be assessed during the data cleaning phase.